# Mô hình mua kèm (upsell) cho chatbot HDX Tech trên TikTok Shop

Shop mới chưa có đơn hàng, nên nhóm học **"loại hàng nào hay được mua cùng nhau"** từ bộ dữ liệu công khai **Amazon Reviews 2023** (McAuley Lab, UCSD), nhóm *Electronics* (laptop, bàn phím, chuột, tai nghe, màn hình, SSD, hub, micro, webcam, lót chuột).

- Một người **đánh giá hai món khác loại trong vòng 30 ngày** được coi là đã mua chúng cùng nhau.
- Đếm theo người để ra **P(loại B | đang mua loại A)**, làm mượt về mức phổ biến chung khi A ít dữ liệu.
- **Đánh giá:** học trên 80% thời gian đầu, kiểm tra trên 20% thời gian sau. Chấm **Hit@1**, **Hit@3** (có ít nhất 1 loại đúng trong 3 gợi ý đầu) và **Recall@5**, so với **baseline** "mời loại bán chạy nhất".
- **So sánh 5 cách gợi ý:** baseline phổ biến, **P(B|A)** (lọc cộng tác theo sản phẩm, cách của mục *Frequently bought together*), **P(B|A) + lọc lift**, **Lift / PMI** (luật kết hợp), và **phân rã ma trận ALS** (implicit feedback, Hu–Koren–Volinsky 2008). Luật nạp vào hệ thống mặc định lấy từ **P(B|A) + lọc lift**: mời loại hàng thật sự đi kèm, không chỉ loại bán chạy.
- **Kết quả:** bảng luật `upsell_rules.sql` để nạp vào database middleware. Chatbot dùng luật này chọn *loại* hàng để mời; món cụ thể do dữ liệu kho lọc (còn hàng, hợp máy); **bandit** chọn món nào trong số đó.

## Cách chạy (Colab, **không cần GPU**)
1. Chép thư mục `upsell/` của repo lên Google Drive vào `MyDrive/cntttmdt/upsell`.
2. Mở notebook này trên Colab. *Runtime type* để **CPU** là đủ.
3. *Runtime > Run all*, cho phép truy cập Drive.
4. Mất khoảng **15–25 phút**, phần lớn để đọc 5 GB thông tin sản phẩm. Kết quả lưu ở `MyDrive/cntttmdt/upsell/outputs/`.

## 1. Cấu hình

In [ ]:
import os, sys, json, time, glob

SMOKE = os.environ.get("UPSELL_SMOKE") == "1"

DATASETS = ["Electronics"]
WINDOW_DAYS = 30
TEST_FRACTION = 0.2
ALPHA = 20.0
MIN_LIFT, MIN_SUPPORT = 1.0, 30
RULE_MODEL = "loc_lift"

MAX_ROWS = 300_000 if SMOKE else None
MAX_META_BYTES = 40_000_000 if SMOKE else None

IN_COLAB = "google.colab" in sys.modules
if os.environ.get("UPSELL_DIR"):
    UPSELL_DIR = os.environ["UPSELL_DIR"]
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    UPSELL_DIR = "/content/drive/MyDrive/cntttmdt/upsell"
else:
    UPSELL_DIR = os.path.abspath(".")
assert os.path.exists(os.path.join(UPSELL_DIR, "upsell_lib.py")), f"Không thấy upsell_lib.py trong {UPSELL_DIR}"
OUT_DIR = os.environ.get("UPSELL_OUT") or os.path.join(UPSELL_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, UPSELL_DIR)

import importlib
import pandas as pd
import upsell_lib as U
U = importlib.reload(U)
dups = [f for f in os.listdir(UPSELL_DIR) if f.startswith("upsell_lib") and f != "upsell_lib.py"]
assert hasattr(U, "LiftFiltered"), (
    "upsell_lib.py trên Drive là bản cũ. Xóa nó (và các bản trùng tên " + str(dups) + "), "
    "tải bản mới từ repo lên, rồi Thời gian chạy > Khởi động lại phiên > Chạy tất cả.")
import category_map as CM
CM = importlib.reload(CM)
from category_map import SHOP_CATEGORIES
print("Thư mục:", UPSELL_DIR, "| kết quả:", OUT_DIR, "| CHẠY THỬ NHỎ" if SMOKE else "")

## 2. Đọc đánh giá (bản 5-core)
Mỗi dòng: một người đánh giá một sản phẩm lúc nào. *5-core*: chỉ giữ người và sản phẩm có ít nhất 5 đánh giá, nên ít nhiễu và nhẹ (160 MB + 900 MB).

In [ ]:
ratings = {cat: U.read_ratings(cat, MAX_ROWS) for cat in DATASETS}

## 3. Gán loại hàng của shop cho từng sản phẩm Amazon
Đọc file thông tin sản phẩm theo từng dòng; chỉ giữ sản phẩm có trong bảng đánh giá. Gán loại bằng nhánh cuối của cây danh mục Amazon (`category_map.py`), ví dụ `Laptops > Traditional Laptops` thành **laptop**, `Data Storage > Internal Solid State Drives` thành **ssd**; túi laptop, giá đỡ màn hình, đệm tai nghe thì bỏ. Món không thuộc loại nào shop bán thì bỏ.

Kết quả được lưu đệm vào `outputs/item_category_<phiên bản bảng gán loại>.json`; chạy lại notebook sẽ không phải đọc lại. Đổi bảng gán loại thì tăng `MAP_VERSION` trong `category_map.py` để đọc lại.

In [ ]:
cache = os.path.join(OUT_DIR, f"item_category_{CM.MAP_VERSION}{'_smoke' if SMOKE else ''}.json")
if os.path.exists(cache):
    item_cat = json.load(open(cache, encoding="utf-8"))
    print(f"Đọc lại từ {cache}: {len(item_cat):,} sản phẩm")
else:
    item_cat = {}
    for cat, df in ratings.items():
        item_cat.update(U.stream_meta(cat, set(df.parent_asin), MAX_META_BYTES))
    json.dump(item_cat, open(cache, "w", encoding="utf-8"))
pd.Series(item_cat).value_counts().rename("số sản phẩm")

## 4. Dựng chuỗi mua hàng theo người, tách học / kiểm tra theo thời gian

In [ ]:
events = pd.concat([U.build_events(df, item_cat) for df in ratings.values()]).drop_duplicates()
events = events.sort_values(["user_id", "day"]).reset_index(drop=True)
cutoff = events.day.quantile(1 - TEST_FRACTION)
train_ev, test_ev = events[events.day <= cutoff], events[events.day > cutoff]
print(f"{len(events):,} lượt mua thuộc loại của shop, {events.user_id.nunique():,} người")
print(f"Học: trước ngày {pd.to_datetime(cutoff, unit='D').date()} ({len(train_ev):,} lượt) | Kiểm tra: sau đó ({len(test_ev):,} lượt)")

## 5. Học: đếm cặp loại hàng mua cùng nhau

In [ ]:
t0 = time.time()
co, anchor, n_users = U.count_pairs(train_ev, WINDOW_DAYS)
cases = U.test_cases(test_ev, WINDOW_DAYS)
print(f"{sum(co.values()):,} cặp (đếm theo người) | {len(cases):,} ca kiểm tra | {time.time() - t0:.0f}s")
baseline, model = U.Popularity(co), U.CoPurchase(co, anchor, ALPHA)
lift = U.Lift(co, anchor, n_users, ALPHA)
loc_lift = U.LiftFiltered(co, anchor, n_users, ALPHA, MIN_LIFT, MIN_SUPPORT)
t0 = time.time()
als = U.ALSMatrix(train_ev, factors=8, reg=0.1, alpha=10.0, iters=12)
print(f"ALS: {als.n_users:,} người có từ 2 loại hàng trở lên | {time.time() - t0:.0f}s")
MODELS = [baseline, model, loc_lift, lift, als]
RULE_SOURCE = {"p_b_a": model, "loc_lift": loc_lift}[RULE_MODEL]

## 6. Đánh giá: so sánh 5 cách gợi ý
- **Baseline (phổ biến):** luôn mời loại bán chạy nhất. Mô hình nào không thắng được baseline thì vô dụng.
- **Mua cùng nhau P(B|A):** lọc cộng tác theo sản phẩm (item-to-item), đếm xem người mua A có mua B hay không.
- **P(B|A) + lọc lift:** như P(B|A), nhưng chỉ ưu tiên loại B được mua kèm A **nhiều hơn mức bình thường** (lift ≥ 1) và có ít nhất 30 người làm căn cứ; loại còn lại xếp sau. Tránh việc mời "ốp lưng" cho mọi thứ chỉ vì ốp lưng bán chạy.
- **Lift / PMI:** P(B|A) chia cho P(B). Thưởng cặp "hợp nhau bất thường", nhưng hay đẩy loại hiếm lên đầu.
- **Phân rã ma trận ALS:** biểu diễn mỗi người và mỗi loại hàng bằng 8 con số ẩn, học từ ma trận ai mua loại nào (không xét cửa sổ 30 ngày), rồi gợi ý cho khách vừa mua A.

**Chỉ số:** Hit@1 (gợi ý đầu tiên đúng), Hit@3, Recall@5. Shop chỉ có 10 loại hàng nên Hit@3 của cách nào cũng cao; Hit@1 và bảng theo từng loại hàng phân biệt rõ hơn.


In [ ]:
RESULTS = {m.name: U.evaluate(m, cases) for m in MODELS}
table = pd.DataFrame(RESULTS).T
display(table) if "display" in globals() else print(table)

import matplotlib.pyplot as plt
ax = table[["hit_at_1", "hit_at_3", "recall_at_5"]].plot.bar(figsize=(8, 4), rot=0, title="So sánh các cách gợi ý mua kèm")
ax.set_ylim(0, 1); ax.set_ylabel("điểm (cao hơn là tốt hơn)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "so_sanh_mo_hinh.png"), dpi=150); plt.show()

per = []
for a in SHOP_CATEGORIES:
    sub = [c for c in cases if c[0] == a]
    if sub:
        row = {"đang mua": a, "số ca": len(sub)}
        row.update({f"Hit@1 {m.name}": U.evaluate(m, sub)["hit_at_1"] for m in MODELS})
        per.append(row)
per_df = pd.DataFrame(per).sort_values("số ca", ascending=False)
per_df


## 6b. Thử cửa sổ ngắn hơn
"Mua cùng nhau" đang tính là hai lần đánh giá cách nhau tối đa 30 ngày. Hai món đánh giá **cùng ngày** gần với "cùng một đơn hàng" hơn, nên có thể thấy rõ món đi kèm thật. Ô này chạy lại với cửa sổ 1 và 7 ngày (mỗi cửa sổ khoảng 2–3 phút) và in 5 cặp có lift cao nhất.


In [ ]:
WINDOWS_THU = [1, 7]
sweep_rows, best_window = [], {}
for w in WINDOWS_THU + [WINDOW_DAYS]:
    t0 = time.time()
    if w == WINDOW_DAYS:
        co_w, anchor_w, n_w, cases_w = co, anchor, n_users, cases
    else:
        co_w, anchor_w, n_w = U.count_pairs(train_ev, w)
        cases_w = U.test_cases(test_ev, w)
    ms = [U.Popularity(co_w), U.CoPurchase(co_w, anchor_w, ALPHA),
          U.LiftFiltered(co_w, anchor_w, n_w, ALPHA, MIN_LIFT, MIN_SUPPORT)]
    for m in ms:
        sweep_rows.append({"cửa sổ (ngày)": w, "cách": m.name, **U.evaluate(m, cases_w)})
    top = sorted(((a, b, ms[2].lift(a, b)) for a in SHOP_CATEGORIES for b in SHOP_CATEGORIES
                  if a != b and co_w[(a, b)] >= MIN_SUPPORT), key=lambda x: -x[2])[:5]
    print(f"cửa sổ {w} ngày: {len(cases_w):,} ca | lift cao nhất: " + ", ".join(f"{a}→{b} {l:.2f}" for a, b, l in top)
          + f" | {time.time() - t0:.0f}s")
sweep = pd.DataFrame(sweep_rows)
sweep


## 6c. Kết luận: mô hình có đạt không
**Đạt** khi cách gợi ý tốt nhất có Hit@3 **cao hơn baseline ít nhất 0,5 điểm** (0,005) trên cùng các ca kiểm tra. Không đạt nghĩa là dữ liệu Amazon không cho thêm thông tin gì ngoài "mời món bán chạy"; khi đó **không nạp** `upsell_rules.sql`, giữ 22 luật mặc định viết tay và để bandit học từ khách thật.


In [ ]:
base = RESULTS[baseline.name]
best_name = max((m.name for m in MODELS if m is not baseline), key=lambda n: (RESULTS[n]["hit_at_3"], RESULTS[n]["hit_at_1"]))
best = RESULTS[best_name]
DAT = best["hit_at_3"] >= base["hit_at_3"] + 0.005
cua_so_tot = None
if "sweep" in globals():
    pba = sweep[sweep["cách"] != baseline.name].copy()
    pba["hon_baseline"] = pba.apply(lambda r: r["hit_at_3"] - sweep[(sweep["cửa sổ (ngày)"] == r["cửa sổ (ngày)"]) & (sweep["cách"] == baseline.name)]["hit_at_3"].iloc[0], axis=1)
    top = pba.sort_values("hon_baseline", ascending=False).iloc[0]
    cua_so_tot = {"cua_so": int(top["cửa sổ (ngày)"]), "cach": top["cách"], "hon_baseline_hit3": round(float(top["hon_baseline"]), 4)}
KET_LUAN = {"dat": bool(DAT), "cach_tot_nhat": best_name, "hit3_tot_nhat": best["hit_at_3"], "hit3_baseline": base["hit_at_3"],
            "cua_so_tot_nhat": cua_so_tot}
print(f"Cách tốt nhất: {best_name} | Hit@3 {best['hit_at_3']} so với baseline {base['hit_at_3']}")
print("ĐẠT: nạp upsell_rules.sql vào hệ thống." if DAT else
      "KHÔNG ĐẠT: không nạp upsell_rules.sql; giữ luật mặc định viết tay, bandit học từ khách thật.")
if cua_so_tot:
    print(f"Cửa sổ tốt nhất: {cua_so_tot['cua_so']} ngày, {cua_so_tot['cach']}, hơn baseline {cua_so_tot['hon_baseline_hit3']:+.4f} Hit@3"
          + (" -> đặt WINDOW_DAYS theo cửa sổ này rồi chạy lại để xuất luật." if cua_so_tot["hon_baseline_hit3"] >= 0.005 and cua_so_tot["cua_so"] != WINDOW_DAYS else ""))


## 7. Hình: xác suất mua kèm P(B | A)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
mat = pd.DataFrame([[model.score(a, b) if a != b else float("nan") for b in SHOP_CATEGORIES] for a in SHOP_CATEGORIES],
                   index=SHOP_CATEGORIES, columns=SHOP_CATEGORIES)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(mat.values, cmap="Blues")
ax.set_xticks(range(len(SHOP_CATEGORIES)), SHOP_CATEGORIES, rotation=45, ha="right")
ax.set_yticks(range(len(SHOP_CATEGORIES)), SHOP_CATEGORIES)
ax.set_xlabel("Mời mua kèm (B)"); ax.set_ylabel("Khách đang mua (A)")
ax.set_title("P(B | A) học từ Amazon Reviews 2023")
fig.colorbar(im)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "xac_suat_mua_kem.png"), dpi=150); plt.show()

## 8. Xuất luật cho hệ thống
Luật lấy từ cách đặt ở `RULE_MODEL` (cell cấu hình): `"loc_lift"` (mặc định) hoặc `"p_b_a"`. Cột `lift` cho biết B được mua kèm A nhiều gấp mấy lần mức bình thường.
- `upsell_rules.sql`: nạp vào database middleware (xem bước 9).
- `upsell_rules.csv`: để xem hoặc đưa vào báo cáo.
- `metrics.json`: số liệu đánh giá.

In [ ]:
rows = U.rules(RULE_SOURCE, k=3)
print("Luật lấy từ:", RULE_SOURCE.name)
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "upsell_rules.csv"), index=False, encoding="utf-8-sig")
open(os.path.join(OUT_DIR, "upsell_rules.sql"), "w", encoding="utf-8").write(U.rules_sql(rows))
json.dump({"smoke": SMOKE, "datasets": DATASETS, "window_days": WINDOW_DAYS, "alpha": ALPHA,
           "rule_model": RULE_SOURCE.name, "min_lift": MIN_LIFT, "min_support": MIN_SUPPORT,
           "events": int(len(events)), "users": int(events.user_id.nunique()), "test_cases": len(cases),
           "results": RESULTS, "ket_luan": KET_LUAN, "time": time.strftime("%Y-%m-%d %H:%M")},
          open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã lưu:", sorted(os.listdir(OUT_DIR)))
pd.DataFrame(rows).pivot(index="anchor_category", columns="rank", values=["rec_category", "lift"])

## 9. Nạp luật vào hệ thống (trên máy trưởng nhóm)
1. Tải `outputs/upsell_rules.sql` từ Drive về thư mục `upsell/outputs/` trong repo.
2. Chạy bằng Git Bash:
   ```bash
   docker compose exec -T db psql -U odoo -d middleware < upsell/outputs/upsell_rules.sql
   ```
3. Chatbot tự dùng luật `amazon` thay cho luật mặc định ngay ở tin nhắn tiếp theo, không cần khởi động lại.

**Giới hạn:** luật học ở mức *loại hàng* (10 loại của HDX Tech), từ thói quen mua của người dùng Amazon (chủ yếu ở Mỹ), và "mua cùng nhau" được suy ra từ đánh giá. Bandit trong middleware sẽ điều chỉnh theo khách thật của shop.